In [1]:
from keras.datasets import mnist
from keras.models import Sequential, Model
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from keras.layers import Input, Add, BatchNormalization
from tensorflow.keras.utils import to_categorical

# =======================
# LOAD DATA
# =======================
(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train = X_train.reshape(-1,28,28,1)/255.0
X_test = X_test.reshape(-1,28,28,1)/255.0

y_train = to_categorical(y_train,10)
y_test = to_categorical(y_test,10)

results = {}

2026-03-25 11:59:14.390325: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774439954.806122      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774439954.913137      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774439955.919819      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774439955.919902      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774439955.919905      24 computation_placer.cc:177] computation placer alr

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


## Step 2: LeNet Architecture

LeNet-5 is one of the earliest Convolutional Neural Networks (CNNs), proposed by Yann LeCun in 1998. It was designed specifically for handwritten digit recognition, making it highly suitable for the MNIST dataset.

### Key Idea:
LeNet extracts features from images using convolutional layers and then performs classification using fully connected layers.

### Architecture Overview:

1. **Input Layer**
   - Input size: 1 × 28 × 28 (grayscale image)

2. **Convolution Layer 1**
   - Filters: 6
   - Kernel size: 5×5
   - Output: 6 × 24 × 24

3. **Average Pooling**
   - Reduces spatial dimensions
   - Output: 6 × 12 × 12

4. **Convolution Layer 2**
   - Filters: 16
   - Kernel size: 5×5
   - Output: 16 × 8 × 8

5. **Average Pooling**
   - Output: 16 × 4 × 4

6. **Flatten Layer**
   - Converts into vector (16×4×4 = 256)

7. **Fully Connected Layers**
   - FC1: 120 neurons
   - FC2: 84 neurons
   - Output: 10 classes

### Activation Function:
ReLU is used instead of tanh (modern improvement).

### Why LeNet?
- Simple and efficient
- Works very well on MNIST
- Foundation for modern CNNs

In [2]:
lenet_model = Sequential([
    Conv2D(6,(5,5),activation='tanh',input_shape=(28,28,1)),
    MaxPooling2D(),
    Conv2D(16,(5,5),activation='tanh'),
    MaxPooling2D(),
    Flatten(),
    Dense(120,activation='tanh'),
    Dense(84,activation='tanh'),
    Dense(10,activation='softmax')
])

lenet_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
lenet_model.fit(X_train, y_train, epochs=2, batch_size=64, verbose=0)

_, acc = lenet_model.evaluate(X_test, y_test, verbose=0)
results['LeNet'] = acc

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1774439995.784275      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774439995.790732      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1774440000.191636      73 service.cc:152] XLA service 0x79ec300061d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774440000.191684      73 servic

In [3]:
print("LeNet Accuracy: ", results['LeNet'])

LeNet Accuracy:  0.9837999939918518


## AlexNet Architecture with Feature Extraction Explanation

AlexNet improves upon LeNet by making the network deeper and introducing more filters, allowing it to learn more complex features.

---

### How Feature Extraction Improves in AlexNet

1. **More Filters (96 → 256 → 384)**
   - Each filter learns a different pattern
   - More filters = more feature diversity

Example:
- One filter detects edges
- Another detects curves
- Another detects textures

---

2. **Deeper Convolution Layers**
   - Layer 1: edges
   - Layer 2: shapes
   - Layer 3: complex structures

This is called **hierarchical feature learning**

---

3. **ReLU Activation**
   - Removes negative values
   - Speeds up training
   - Helps model learn faster and deeper patterns

---

4. **MaxPooling (Focus on Strong Features)**
   - Keeps strongest signals
   - Ignores weak/noisy features

Helps model focus on important regions

---

5. **Dropout (Better Generalization)**
   - Randomly disables neurons
   - Forces model to learn robust features

---

### Why AlexNet is Better than LeNet
- More layers → deeper understanding
- More filters → richer features
- Better generalization

---

### Key Insight
LeNet learns “what a digit looks like”  
AlexNet learns “many ways a digit can look”

In [4]:
alex_model = Sequential([
    Conv2D(96,(3,3),activation='relu',input_shape=(28,28,1)),
    MaxPooling2D(),
    Conv2D(256,(3,3),activation='relu'),
    MaxPooling2D(),
    Conv2D(384,(3,3),activation='relu'),
    Flatten(),
    Dense(256,activation='relu'),
    Dropout(0.1),
    Dense(10,activation='softmax')
])

alex_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
alex_model.fit(X_train, y_train, epochs=2, batch_size=64, verbose=0)

_, acc = alex_model.evaluate(X_test, y_test, verbose=0)
results['AlexNet'] = acc


In [5]:
print("Alex Net Accuracy: ", results['AlexNet'])

Alex Net Accuracy:  0.9873999953269958


## ZFNet Architecture with Feature Extraction Explanation

ZFNet improves AlexNet by tuning hyperparameters like filter size and stride to preserve more information.

---

### How Feature Extraction Improves in ZFNet

1. **Smaller Filters (3×3 instead of large kernels)**
   - Capture fine details
   - Preserve spatial information

 Large filters = lose detail  
 Small filters = capture precise patterns

---

2. **Smaller Stride**
   - Filters move slowly across image
   - More overlap → better learning

 More detailed feature maps

---

3. **Better Feature Visualization**
   - ZFNet introduced techniques to visualize what the network learns
   - Helps ensure meaningful feature extraction

---

4. **Deeper Network**
   - More layers → more abstraction levels

---

### Why ZFNet is Better than AlexNet
- Captures finer details
- Reduces information loss
- More accurate feature representation

---

### Key Insight
AlexNet: “Detect features”  
ZFNet: “Detect features more precisely”

In [6]:
zf_model = Sequential([
    Conv2D(96,(3,3),activation='relu',input_shape=(28,28,1)),
    MaxPooling2D(),
    Conv2D(256,(3,3),activation='relu'),
    MaxPooling2D(),
    Conv2D(384,(3,3),activation='relu'),
    Flatten(),
    Dense(256,activation='relu'),
    Dense(10,activation='softmax')
])

zf_model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['accuracy'])
zf_model.fit(X_train, y_train, epochs=10, batch_size=64, verbose=0)

_, acc = zf_model.evaluate(X_test, y_test, verbose=0)
results['ZF-Net'] = acc

In [7]:
print("ZF Net Accuracy: ", results['ZF-Net'])

ZF Net Accuracy:  0.9886000156402588


## VGGNet Architecture with Feature Extraction Explanation

VGGNet focuses on increasing depth using very small (3×3) convolution filters.

---

### How Feature Extraction Improves in VGGNet

1. **Multiple 3×3 Convolutions Instead of Large Filters**
   - Two 3×3 ≈ one 5×5
   - But with more non-linearity

👉 More layers = more learning capacity

---

2. **Deep Architecture**
   - Many stacked layers
   - Each layer refines features further

---

3. **Hierarchical Feature Learning**
   - Early layers: edges
   - Middle layers: shapes
   - Deep layers: full object structure

---

4. **Uniform Design**
   - Same filter size everywhere
   - Easier optimization

---

### Why VGG is Better
- Very deep → learns complex patterns
- More non-linear transformations

---

### Key Insight
LeNet: shallow learning  
AlexNet: deeper learning  
VGG: **very deep feature refinement**

In [8]:
vgg_model = Sequential([
    Conv2D(64,(3,3),activation='relu',padding='same',input_shape=(28,28,1)),
    Conv2D(64,(3,3),activation='relu',padding='same'),
    MaxPooling2D(),
    Conv2D(128,(3,3),activation='relu',padding='same'),
    Conv2D(128,(3,3),activation='relu',padding='same'),
    MaxPooling2D(),
    Flatten(),
    Dense(256,activation='relu'),
    Dense(10,activation='softmax')
])

vgg_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
vgg_model.fit(X_train, y_train, epochs=2, batch_size=64, verbose=0)

_, acc = vgg_model.evaluate(X_test, y_test, verbose=0)
results['VGGNet'] = acc

In [9]:
print("VGGNet Accuracy: ", results['VGGNet'])

VGGNet Accuracy:  0.9894000291824341


## GoogLeNet Architecture with Feature Extraction Explanation

GoogLeNet introduces the Inception module, which processes multiple filter sizes in parallel.

---

### How Feature Extraction Improves in GoogLeNet

1. **Multi-Scale Processing**
   - 1×1 → fine details
   - 3×3 → medium features
   - 5×5 → large patterns

Model sees features at different scales simultaneously

---

2. **Parallel Feature Extraction**
   - Instead of choosing one filter size
   - Uses all of them at once

Captures more diverse features

---

3. **1×1 Convolutions (Dimensionality Reduction)**
   - Reduce number of channels
   - Decrease computation

Efficient but powerful

---

4. **Global Average Pooling**
   - Replaces large fully connected layers
   - Reduces overfitting

---

### Why GoogLeNet is Better
- Captures features at multiple scales
- More efficient than VGG
- Fewer parameters but high performance

---

### Key Insight
Other models: “Choose one way to see features”  
GoogLeNet: “See features in all possible ways”

In [10]:
input_layer = Input(shape=(28,28,1))

c1 = Conv2D(32,(1,1),activation='relu',padding='same')(input_layer)
c3 = Conv2D(32,(3,3),activation='relu',padding='same')(input_layer)
c5 = Conv2D(32,(5,5),activation='relu',padding='same')(input_layer)

merge = MaxPooling2D()(c1 + c3 + c5)
flat = Flatten()(merge)
dense = Dense(128,activation='relu')(flat)
output = Dense(10,activation='softmax')(dense)

google_model = Model(inputs=input_layer, outputs=output)

google_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
google_model.fit(X_train, y_train, epochs=2, batch_size=64, verbose=0)

_, acc = google_model.evaluate(X_test, y_test, verbose=0)
results['GoogLeNet'] = acc

In [11]:
print("GoogLeNet Accuracy: ", results['GoogLeNet'])

GoogLeNet Accuracy:  0.984000027179718


## Comparative Study of CNN Architectures: LeNet, AlexNet, ZFNet, VGGNet, and GoogLeNet

This section presents the original architectures of five landmark convolutional neural networks and analyzes the architectural improvements introduced in each model, along with their impact on feature extraction and performance.

---

## 1. LeNet-5 (1998)

### Architecture (Original Paper)
- Input: 32×32 grayscale image
- C1: Conv (6 filters, 5×5) → 28×28
- S2: Avg Pooling (2×2) → 14×14
- C3: Conv (16 filters, 5×5) → 10×10
- S4: Avg Pooling (2×2) → 5×5
- C5: Conv (120 filters, 5×5) → 1×1
- F6: Fully Connected (84 units)
- Output: 10 classes

### Parameters
- Total parameters: ~60,000

### Key Contribution
- Introduced convolution + pooling for spatial feature extraction

### Limitation
- Shallow network → limited hierarchical feature learning
- Uses tanh activation → slow convergence

---

## 2. AlexNet (2012)

### Architecture (Original Paper)
- Input: 227×227×3
- Conv1: 96 filters (11×11, stride 4)
- MaxPool
- Conv2: 256 filters (5×5)
- MaxPool
- Conv3: 384 filters (3×3)
- Conv4: 384 filters (3×3)
- Conv5: 256 filters (3×3)
- MaxPool
- FC1: 4096
- FC2: 4096
- Output: 1000 classes

### Parameters
- Total parameters: ~60 million

### Improvements over LeNet
1. **Increased Depth (5 conv layers vs 2)**
   → Enabled hierarchical feature learning  
   → Result: Ability to learn complex patterns (textures, object parts)

2. **ReLU Activation**
   → Replaced tanh  
   → Result: Faster convergence (non-saturating gradients)

3. **Max Pooling instead of Average Pooling**
   → Retains strongest activations  
   → Result: Better feature localization

4. **Dropout (in FC layers)**
   → Reduces co-adaptation  
   → Result: Prevents overfitting

5. **GPU Training**
   → Enabled large-scale training

### Impact
- Significant reduction in ImageNet error rate (~15%)
- Established deep CNNs as dominant models

---

## 3. ZFNet (2013)

### Architecture (Modification of AlexNet)
- Similar structure to AlexNet
- Key changes:
  - Conv1: 7×7 filters (instead of 11×11)
  - Reduced stride (from 4 → 2)
  - Improved hyperparameters

### Parameters
- ~60 million (similar scale to AlexNet)

### Improvements over AlexNet
1. **Smaller Filter Size in Early Layers**
   → 11×11 → 7×7  
   → Result: Reduced information loss in early feature maps

2. **Reduced Stride**
   → More overlap during convolution  
   → Result: Higher spatial resolution in feature maps

3. **Feature Visualization (Deconvolution)**
   → Enabled inspection of learned features  
   → Result: Better architecture tuning

### Impact
- Improved top-5 accuracy in ImageNet
- Demonstrated importance of preserving spatial information

---

## 4. VGGNet (2014)

### Architecture (VGG-16)
- Input: 224×224×3
- Conv blocks:
  - (2× Conv 64) + Pool
  - (2× Conv 128) + Pool
  - (3× Conv 256) + Pool
  - (3× Conv 512) + Pool
  - (3× Conv 512) + Pool
- FC1: 4096
- FC2: 4096
- Output: 1000 classes

### Parameters
- ~138 million parameters

### Improvements over ZFNet
1. **Use of Small Filters (3×3 throughout)**
   → Replaced larger filters (5×5, 7×7)  
   → Result:
     - Same receptive field with fewer parameters
     - More non-linearities → richer feature representation

2. **Increased Depth (16–19 layers)**
   → Deeper hierarchical feature extraction  
   → Result: Ability to model highly complex structures

3. **Uniform Architecture Design**
   → Simplifies optimization and reproducibility

### Impact
- Significant improvement in accuracy
- Became standard backbone for many tasks

### Limitation
- Extremely high parameter count → memory intensive

---

## 5. GoogLeNet (Inception v1, 2014)

### Architecture
- Input: 224×224×3
- 22 layers deep
- Uses **Inception Modules**

### Inception Module Structure
Parallel operations:
- 1×1 conv
- 3×3 conv
- 5×5 conv
- Max pooling
→ Outputs concatenated

### Parameters
- ~5 million parameters (much lower than VGG)

### Improvements over VGGNet
1. **Multi-Scale Feature Extraction**
   → Parallel filters (1×1, 3×3, 5×5)  
   → Result: Captures features at different spatial scales simultaneously

2. **1×1 Convolutions (Dimensionality Reduction)**
   → Reduce channel depth before expensive operations  
   → Result:
     - Lower computational cost
     - Fewer parameters

3. **Global Average Pooling (GAP)**
   → Replaces large FC layers  
   → Result:
     - Reduces overfitting
     - Eliminates millions of parameters

4. **Sparse Connectivity Approximation**
   → Efficient dense implementation  
   → Result: High performance with low computation

### Impact
- State-of-the-art performance with drastically fewer parameters
- Introduced architectural efficiency as a key design principle

---

## Summary of Architectural Evolution

| Model      | Depth | Parameters | Key Improvement                  | Impact |
|-----------|------|-----------|----------------------------------|--------|
| LeNet     | 5    | ~60K      | CNN introduction                 | Basic feature learning |
| AlexNet   | 8    | ~60M      | Depth + ReLU + Dropout           | Deep feature learning |
| ZFNet     | 8    | ~60M      | Better filter/stride tuning      | Improved feature precision |
| VGGNet    | 16–19| ~138M     | Small filters + deeper network   | Rich feature representation |
| GoogLeNet | 22   | ~5M       | Inception modules + efficiency   | Multi-scale + efficient learning |

---

## Conclusion

The evolution of CNN architectures shows a clear progression:
- Increasing depth improves representational power
- Smaller filters improve feature precision
- Parallel architectures improve multi-scale understanding
- Parameter efficiency becomes critical in deeper networks

Each model builds upon its predecessor by addressing specific limitations, leading to improved accuracy, better feature extraction, and optimized computation.